# Exercício 06 — Debugging de Código ML
### Mastering Machine Learning Advanced · ML Engineer Track
### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

---

## Por que isso importa para um ML Engineer?

Em produção, pipelines com bugs **rodam sem erros** mas produzem resultados silenciosamente errados. Diagnosticar data leakage, métricas mal calculadas, splits incorretos e encodings invertidos é uma habilidade crítica que diferencia um ML Engineer de um usuário de notebook.

Cada exercício abaixo contém código quebrado. Seu trabalho é:
1. Identificar **o que está errado** e **por que é um problema**
2. Corrigir o código
3. Verificar que a correção muda o resultado de forma esperada

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# dataset base
url = 'https://raw.githubusercontent.com/ahirtonlopes/Mastering-Machine-Learning/main/Bases/TelcoChurn.csv'
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df = df.drop(columns=['customerID'])
print(f'Dataset: {df.shape}')

---
## Bug 6.1 — Data Leakage no Pré-processamento

O código abaixo reporta uma acurácia suspeitamente alta. Encontre o bug, explique por que ele é data leakage, e corrija-o.

In [ ]:
# === CÓDIGO BUGADO — NÃO ALTERE ESTA CÉLULA ===
cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_bugado = df[cols].values
y_bugado = df['Churn'].values

# normalização ANTES do split
scaler_bug = StandardScaler()
X_scaled_bug = scaler_bug.fit_transform(X_bugado)  # BUG AQUI

X_tr_bug, X_te_bug, y_tr_bug, y_te_bug = train_test_split(
    X_scaled_bug, y_bugado, test_size=0.2, random_state=42
)

lr_bug = LogisticRegression()
lr_bug.fit(X_tr_bug, y_tr_bug)
acc_bug = accuracy_score(y_te_bug, lr_bug.predict(X_te_bug))
print(f'Acurácia (código bugado): {acc_bug:.4f}')

In [ ]:
# === SUA CORREÇÃO AQUI ===
# 1. Explique o bug em um comentário
# 2. Corrija o código
# 3. Compare a acurácia correta vs. bugada

# EXPLICAÇÃO DO BUG:
# ...

# CÓDIGO CORRIGIDO:
# ...

# --- VALIDAÇÃO ---
# A diferença de acurácia deve ser pequena (leakage numérico não muda muito aqui)
# mas o processo correto é o que importa para dados com distribuição muito diferente
# assert acc_correto <= acc_bug + 0.02, 'Acurácia corrigida não deve ser muito maior'
# print(f'Acurácia correta: {acc_correto:.4f} | Bugada: {acc_bug:.4f}')
# print('✅ Bug 6.1 corrigido!')

---
## Bug 6.2 — Métrica Errada para Dataset Desbalanceado

O dataset de churn tem ~26% de positivos. O código abaixo usa accuracy e conclui que o modelo é excelente. Identifique o problema, mostre por que accuracy engana aqui, e use as métricas corretas.

In [ ]:
# === CÓDIGO BUGADO — NÃO ALTERE ESTA CÉLULA ===
from sklearn.dummy import DummyClassifier

y_all = df['Churn'].values
print(f'Proporção de churn: {y_all.mean():.2%}')

# modelo dummy que sempre prevê a classe majoritária
dummy = DummyClassifier(strategy='most_frequent')
X_dummy = df[['tenure', 'MonthlyCharges']].values
dummy.fit(X_dummy, y_all)
y_pred_dummy = dummy.predict(X_dummy)

acc_dummy = accuracy_score(y_all, y_pred_dummy)
print(f'Accuracy do modelo dummy: {acc_dummy:.4f}')  # vai parecer ótimo!
print('Conclusão bugada: nosso modelo tem 73% de acurácia, ótimo!')

In [ ]:
# === SUA ANÁLISE E CORREÇÃO AQUI ===
# 1. Mostre que o dummy classifier é inútil usando métricas adequadas
# 2. Treine um modelo real e compare usando precision, recall, F1, AUC-ROC
# 3. Explique qual métrica é mais relevante para o problema de churn

# MÉTRICAS NO DUMMY:
# ...

# MODELO REAL COM MÉTRICAS CORRETAS:
# ...

# --- VALIDAÇÃO ---
# f1_dummy deve ser muito próximo de 0
# assert f1_score(y_all, y_pred_dummy, zero_division=0) < 0.01
# print('✅ Bug 6.2 identificado e corrigido!')

---
## Bug 6.3 — Label Encoding em Variável Nominal

O código abaixo aplica `LabelEncoder` em uma variável categórica nominal (sem ordem). Isso cria um problema sutil que afeta modelos baseados em distância e regressão. Identifique, explique e corrija.

In [ ]:
# === CÓDIGO BUGADO — NÃO ALTERE ESTA CÉLULA ===
df_enc_bug = df[['Contract', 'MonthlyCharges', 'Churn']].copy()

# Contract tem valores: 'Month-to-month', 'One year', 'Two year'
print('Valores de Contract:', df_enc_bug['Contract'].unique())

le_bug = LabelEncoder()
df_enc_bug['Contract_enc'] = le_bug.fit_transform(df_enc_bug['Contract'])  # BUG
print('Encoding aplicado:', dict(zip(le_bug.classes_, le_enc_bug.transform(le_bug.classes_))))

# agora o modelo vai entender que 'Two year' > 'One year' > 'Month-to-month'
# o que é uma ordem arbitrária criada pelo LabelEncoder
X_enc_bug = df_enc_bug[['Contract_enc', 'MonthlyCharges']].values
y_enc = df_enc_bug['Churn'].values

In [ ]:
# === SUA CORREÇÃO AQUI ===
# 1. Explique por que LabelEncoder é errado para variáveis nominais
# 2. Use OneHotEncoder ou pd.get_dummies para corrigir
# 3. Treine Logistic Regression nas duas versões e compare AUC-ROC

# EXPLICAÇÃO:
# ...

# CÓDIGO CORRIGIDO:
# ...

# --- VALIDAÇÃO ---
# assert auc_correto >= auc_bugado - 0.01, 'Encoding correto deve ter AUC >= bugado'
# print('✅ Bug 6.3 corrigido!')

---
## Bug 6.4 — Cross-Validation com Leakage de Transformação

O código abaixo faz cross-validation, mas a normalização está fora do Pipeline, o que causa leakage em cada fold. A acurácia reportada é otimista. Identifique, explique e corrija usando Pipeline.

In [ ]:
# === CÓDIGO BUGADO — NÃO ALTERE ESTA CÉLULA ===
from sklearn.model_selection import StratifiedKFold

cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_cv = df[cols_num].values
y_cv = df['Churn'].values

# LEAKAGE: scaler é fitado em TODOS os dados antes do CV
scaler_cv = StandardScaler()
X_cv_scaled = scaler_cv.fit_transform(X_cv)  # BUG: usa todos os dados

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_bug = cross_val_score(
    LogisticRegression(), X_cv_scaled, y_cv, cv=cv, scoring='roc_auc'
)
print(f'AUC-ROC (bugado):  {scores_bug.mean():.4f} ± {scores_bug.std():.4f}')

In [ ]:
# === SUA CORREÇÃO AQUI ===
# Use sklearn Pipeline para encapsular StandardScaler + LogisticRegression
# Passe o pipeline diretamente para cross_val_score

# CÓDIGO CORRIGIDO:
# pipeline_correto = Pipeline([...])
# scores_correto = cross_val_score(...)

# --- VALIDAÇÃO ---
# A diferença deve ser pequena neste dataset, mas o processo correto é o que importa
# print(f'AUC-ROC (correto): {scores_correto.mean():.4f} ± {scores_correto.std():.4f}')
# assert scores_correto.mean() > 0.75
# print('✅ Bug 6.4 corrigido!')

---
## Bug 6.5 — Target Leakage (o mais perigoso)

O código abaixo inclui uma feature que só existe **porque** o cliente cancelou. Em produção, essa feature não estaria disponível no momento da predição. O modelo parece perfeito — mas é inútil.

Identifique a feature vazada, remova-a, e avalie o modelo sem ela.

In [ ]:
# === CÓDIGO BUGADO — NÃO ALTERE ESTA CÉLULA ===

# Simulando um dataset com target leakage
df_leak = df[['tenure', 'MonthlyCharges', 'Contract', 'Churn']].copy()

# 'cancelamento_registrado' é preenchido pelo sistema CRM APÓS o churn ocorrer
# em produção, clientes ativos têm essa coluna = 0 (ou NaN)
np.random.seed(42)
df_leak['cancelamento_registrado'] = df_leak['Churn'].copy()  # LEAKAGE: é o target!
# com pequeno ruído para disfarçar
noise_idx = np.random.choice(len(df_leak), size=50, replace=False)
df_leak.loc[noise_idx, 'cancelamento_registrado'] = 1 - df_leak.loc[noise_idx, 'cancelamento_registrado']

X_leak = df_leak[['tenure', 'MonthlyCharges', 'cancelamento_registrado']].values  # BUG
y_leak = df_leak['Churn'].values

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y_leak, test_size=0.2, random_state=42)
rf_leak = RandomForestClassifier(n_estimators=50, random_state=42)
rf_leak.fit(X_tr_l, y_tr_l)
auc_leak = roc_auc_score(y_te_l, rf_leak.predict_proba(X_te_l)[:, 1])
print(f'AUC-ROC (com leakage): {auc_leak:.4f}')  # suspeitamente alto!

In [ ]:
# === SUA ANÁLISE E CORREÇÃO AQUI ===
# 1. Explique por que 'cancelamento_registrado' é target leakage
# 2. Remova a feature e retreine
# 3. Compare o AUC-ROC com e sem leakage
# 4. Explique como você detectaria esse problema em dados reais de produção

# EXPLICAÇÃO:
# ...

# CÓDIGO CORRIGIDO:
# ...

# --- VALIDAÇÃO ---
# assert auc_sem_leakage < auc_leak - 0.05, 'Remover leakage deve reduzir AUC significativamente'
# print(f'AUC sem leakage: {auc_sem_leakage:.4f} vs. com leakage: {auc_leak:.4f}')
# print('✅ Bug 6.5 identificado e corrigido!')

---
## Bug 6.6 — Desafio Final: Pipeline com Múltiplos Bugs

O pipeline abaixo tem **3 bugs simultâneos**. Encontre todos, explique cada um e entregue uma versão corrigida que passe na validação.

In [ ]:
# === PIPELINE BUGADO — NÃO ALTERE ESTA CÉLULA ===
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

cols_num2 = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat2 = ['Contract', 'PaymentMethod', 'InternetService']

X_full = df[cols_num2 + cols_cat2]
y_full = df['Churn']

# BUG 1: split DEPOIS do fit do preprocessor (leakage)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), cols_num2),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cols_cat2)
])
X_transformed = preprocessor.fit_transform(X_full)  # BUG 1
X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(
    X_transformed, y_full, test_size=0.2, random_state=42
)

# BUG 2: avaliando no conjunto de treino
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_tr_f, y_tr_f)
auc_full = roc_auc_score(y_tr_f, rf_full.predict_proba(X_tr_f)[:, 1])  # BUG 2
print(f'AUC-ROC (treino, bugado): {auc_full:.4f}')

# BUG 3: usando accuracy para dataset desbalanceado sem ajuste
acc_full = accuracy_score(y_te_f, rf_full.predict(X_te_f))
print(f'Accuracy (teste, bugado): {acc_full:.4f}')  # BUG 3: métrica inadequada

In [ ]:
# === SUA CORREÇÃO AQUI ===
# Identifique os 3 bugs, corrija e valide

# BUG 1 (descreva): ...
# BUG 2 (descreva): ...
# BUG 3 (descreva): ...

# PIPELINE CORRIGIDO:
# ...

# --- VALIDAÇÃO ---
# assert 0.75 < auc_correto_teste < 0.98, f'AUC esperado entre 0.75 e 0.98, obtido {auc_correto_teste:.4f}'
# print(f'AUC-ROC no TESTE (correto): {auc_correto_teste:.4f}')
# print('✅ Bug 6.6 — todos os bugs corrigidos!')

**Reflexão:** dos 6 bugs acima, qual você considera mais perigoso em produção e por quê? Quais estratégias de revisão de código ou testes você implementaria para preveni-los?

*(Escreva sua resposta aqui)*

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)